# Comparing Advantage Estimators

This notebook demonstrates the three CAPO advantage estimators:

1. **`capo`** — Standard CAPO with fixed weights
2. **`capo_l_capo`** — EB estimation of β only (fast)
3. **`capo_eb`** — Full EB with (β, ρ, η) estimation

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np

from capo.verl_integration.adv_estimators import (
    compute_capo_advantage,
    compute_capo_l_capo_advantage,
    compute_capo_eb_full_advantage,
)

## Generate Synthetic Data

We'll create synthetic token-level rewards that mimic a reasoning task where:
- Longer trajectories have higher variance
- There's some within-trajectory correlation

In [ ]:
torch.manual_seed(42)

# 8 trajectories with varying lengths
B = 8
max_len = 100
true_beta = 0.8  # variance scales as L^0.8

# Generate random lengths
lengths = torch.randint(20, max_len, (B,))
print(f"Trajectory lengths: {lengths.tolist()}")

# Create rewards with length-dependent variance
rewards = torch.zeros(B, max_len)
mask = torch.zeros(B, max_len, dtype=torch.long)

for i in range(B):
    L = lengths[i].item()
    # Variance proportional to L^beta
    std = (L ** (true_beta / 2)) * 0.1
    # Add some correlation between adjacent tokens
    noise = torch.randn(L) * std
    for t in range(1, L):
        noise[t] = 0.3 * noise[t-1] + 0.7 * noise[t]  # AR(1) correlation
    rewards[i, :L] = noise
    # Terminal reward (outcome)
    rewards[i, L-1] += torch.randn(1).item() * 0.5
    mask[i, :L] = 1

print(f"Rewards shape: {rewards.shape}")
print(f"Mask shape: {mask.shape}")

## Run All Three Estimators

In [ ]:
# 1. Standard CAPO (fixed beta=1, no dependence modeling)
adv_capo, ret_capo, stats_capo = compute_capo_advantage(
    token_level_rewards=rewards,
    response_mask=mask,
    index=None,
    config=None,
)
print("✓ compute_capo_advantage")

# 2. L-CAPO (estimates beta only)
adv_l_capo, ret_l_capo, stats_l_capo = compute_capo_l_capo_advantage(
    token_level_rewards=rewards,
    response_mask=mask,
    index=None,
    config=None,
)
print("✓ compute_capo_l_capo_advantage")

# 3. Full EB (estimates beta, rho, eta)
adv_eb_full, ret_eb_full, stats_eb_full = compute_capo_eb_full_advantage(
    token_level_rewards=rewards,
    response_mask=mask,
    index=None,
    config=None,
    increments=rewards,  # Use rewards as increments for this demo
    increments_mask=mask,
    beta_steps=5,
    xi_steps=3,
)
print("✓ compute_capo_eb_full_advantage")

## Compare Advantage Distributions

In [ ]:
# Get per-trajectory advantages (average over valid tokens)
def get_trajectory_advantages(adv, mask):
    valid = mask > 0
    return [(adv[i][valid[i]].mean().item()) for i in range(len(adv))]

traj_adv_capo = get_trajectory_advantages(adv_capo, mask)
traj_adv_l_capo = get_trajectory_advantages(adv_l_capo, mask)
traj_adv_eb_full = get_trajectory_advantages(adv_eb_full, mask)

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

x = np.arange(B)
width = 0.25

# Advantages by trajectory
axes[0].bar(x - width, traj_adv_capo, width, label='CAPO', alpha=0.8)
axes[0].bar(x, traj_adv_l_capo, width, label='L-CAPO', alpha=0.8)
axes[0].bar(x + width, traj_adv_eb_full, width, label='EB-Full', alpha=0.8)
axes[0].set_xlabel('Trajectory')
axes[0].set_ylabel('Mean Advantage')
axes[0].set_title('Advantage per Trajectory')
axes[0].legend()
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)

# Advantages vs length
axes[1].scatter(lengths.numpy(), traj_adv_capo, label='CAPO', alpha=0.7, s=100)
axes[1].scatter(lengths.numpy(), traj_adv_l_capo, label='L-CAPO', alpha=0.7, s=100)
axes[1].scatter(lengths.numpy(), traj_adv_eb_full, label='EB-Full', alpha=0.7, s=100)
axes[1].set_xlabel('Trajectory Length')
axes[1].set_ylabel('Mean Advantage')
axes[1].set_title('Advantage vs Length')
axes[1].legend()
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)

# Distribution of advantages
axes[2].hist(traj_adv_capo, bins=10, alpha=0.5, label='CAPO')
axes[2].hist(traj_adv_l_capo, bins=10, alpha=0.5, label='L-CAPO')
axes[2].hist(traj_adv_eb_full, bins=10, alpha=0.5, label='EB-Full')
axes[2].set_xlabel('Advantage')
axes[2].set_ylabel('Count')
axes[2].set_title('Advantage Distribution')
axes[2].legend()

plt.tight_layout()
plt.show()

## Examine EB Statistics

In [ ]:
print("L-CAPO Statistics:")
print(f"  Estimated β: {stats_l_capo.get('beta_hat', 'N/A')}")
print(f"  True β: {true_beta}")
print()
print("EB-Full Statistics:")
print(f"  Estimated β: {stats_eb_full.get('beta_hat', 'N/A')}")
print(f"  Estimated ρ: {stats_eb_full.get('rho_hat', 'N/A')}")
print(f"  Estimated η: {stats_eb_full.get('eta_hat', 'N/A')}")

## Summary

| Estimator | Estimates | Speed | Use Case |
|-----------|-----------|-------|----------|
| `capo` | None (fixed β=1) | Fastest | Baseline, when ΔL is known to work |
| `capo_l_capo` | β only | Fast | Most cases, good default |
| `capo_eb` | β, ρ, η | Slower | Long contexts with token dependence |